In [1]:
import os, warnings 
warnings.filterwarnings("ignore")
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import os
import sys


ENV_CONFIG = {
    "LANGSMITH_TRACING": {"label": "LangSmith Tracing", "is_flag": True},
    "LANGSMITH_API_KEY": {"label": "LangSmith API Key", "is_secret": True},
    "LANGSMITH_PROJECT": {
        "label": "LangSmith Project",
        "default": "default",
    },
    "GROQ_API_KEY": {"label": "Groq API Key", "is_secret": True},
}


def print_env_status():
    """Validates and prints environment variable status in a structured ASCII format."""
    header = "ENVIRONMENT CONFIGURATION CHECK"
    width = 65

    print("\n" + "=" * width)
    print(f" {header.center(width - 2)} ")
    print("=" * width)
    print(f"{'VARIABLE / SERVICE':<25} | {'STATUS / VALUE':<20} | {'STATE':<10}")
    print("-" * width)

    all_critical_passed = True

    for env_var, meta in ENV_CONFIG.items():
        val = os.getenv(env_var)
        label = meta.get("label", env_var)

        if meta.get("is_secret"):
            if val:
                status_str = "Loaded (Masked)"
                state = "[ PASS ]"
            else:
                status_str = "Missing Key"
                state = "[ FAIL ]"
                all_critical_passed = False

        elif meta.get("is_flag"):
            if val and val.lower() in ("true", "1", "yes"):
                status_str = "Enabled"
                state = "[ ACTIVE ]"
            elif val and val.lower() in ("false", "0", "no"):
                status_str = "Disabled"
                state = "[ OFF ]"
            else:
                status_str = f"Unset ({val if val else 'default'})"
                state = "[ WARN ]"

        else:
            default_val = meta.get("default", "N/A")
            status_str = val if val else f"{default_val} (default)"
            state = "[ INFO ]"

        print(f"{label:<25} | {status_str:<20} | {state:<10}")

    print("=" * width)

    if all_critical_passed:
        print(" SUCCESS: All critical API keys and configurations loaded.")
    else:
        print(" WARNING: Missing required API keys. Services may fail.")

    print("=" * width + "\n")


if __name__ == "__main__":
    print_env_status()


                 ENVIRONMENT CONFIGURATION CHECK                 
VARIABLE / SERVICE        | STATUS / VALUE       | STATE     
-----------------------------------------------------------------
LangSmith Tracing         | Enabled              | [ ACTIVE ]
LangSmith API Key         | Loaded (Masked)      | [ PASS ]  
LangSmith Project         | langsmith-lab        | [ INFO ]  
Groq API Key              | Loaded (Masked)      | [ PASS ]  
 SUCCESS: All critical API keys and configurations loaded.



In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0.7)

# LangSmith intercepts this call automatically via the three env vars we set.
response = llm.invoke("What is a LangSmith Run? Answer in 2 sentences.")
print(response.content)